# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring a dataset defined by a Croissant schema using the `mlcroissant` library. It demonstrates how to access and analyze record sets, fields, and data in the FAIR² dataset on rangeland management.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records from the FAIR² Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Output dataset metadata
meta = dataset.metadata
print(f"Title: {meta.name}")
print(f"Identifier: {meta.identifier}")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print("\nDescription:")
print(meta.description)

## 2. Data Overview

Let's review the available record sets, fields, and their corresponding `@id`s. Each record set, field, and column should be referenced by its `@id` according to the Croissant model.

In [ ]:
# List all available record sets, their IDs, and summarized fields
record_sets = list(dataset.record_sets)

print("Record Sets Present in the Dataset:")
for record_set in record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {getattr(record_set, 'description', '(no description)')}")
    print("  Fields (by @id):")
    for field in record_set.fields:
        print(f"    - {field.id} (name: {getattr(field, 'name', '-')})")
    print()

## 3. Data Extraction

We load records from each record set into individual DataFrames for analysis. Record set and field entities are always referenced by their `@id` for programmatic consistency.

In [ ]:
# Prepare DataFrames for all record sets (using `@id` references)
dataframes = {}
for record_set in record_sets:
    rec_id = record_set.id
    # Records generator for this record set
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records from RecordSet {rec_id}")
    if len(df.columns) > 0:
        print(f"Columns in {rec_id}: {df.columns.tolist()}\n")

# For demonstration, pick the first non-empty record set
selected_record_set = None
for rs_id, df in dataframes.items():
    if not df.empty:
        selected_record_set = rs_id
        break

if selected_record_set:
    print(f"Example records from RecordSet {selected_record_set}:")
    display(dataframes[selected_record_set].head())
else:
    print("No records found in available record sets.")

## 4. Exploratory Data Analysis (EDA)

We demonstrate filtering, normalization, and grouping operations on numeric fields from a selected record set. All references use the entity `@id`.

In [ ]:
# EDA on a numeric field (using @id references)
import numpy as np

if selected_record_set:
    df = dataframes[selected_record_set]

    # Find numeric-like columns by @id
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        print("No numeric fields found in the selected record set.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field for analysis (@id): {numeric_field_id}")
        # Filter by a threshold (e.g., mean)
        thresh = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > thresh].copy()
        print(f"Filtered records with {numeric_field_id} > {thresh:.2f} (showing up to 5):")
        print(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized values for {numeric_field_id}:\n")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by another field (for demonstration, pick first categorical field)
        cat_fields = [col for col in df.columns if df[col].dtype == 'O' and col != numeric_field_id]
        group_field_id = cat_fields[0] if cat_fields else None
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped {numeric_field_id} mean by {group_field_id} (@id):")
            print(grouped.head())
        else:
            print("No suitable categorical field available for grouping.")
else:
    print("No usable record set loaded for EDA.")

## 5. Visualization

We'll visualize the distribution of the selected numeric field, and if grouped data is available, show a mean by group. This leverages the `@id` for all field references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and 'numeric_field_id' in locals():
    df = dataframes[selected_record_set]
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax[0])
    ax[0].set_title(f'Distribution of {numeric_field_id}')

    if 'group_field_id' in locals() and group_field_id:
        grouped_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(data=grouped_means, x=group_field_id, y=numeric_field_id, ax=ax[1])
        ax[1].set_title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.setp(ax[1].xaxis.get_majorticklabels(), rotation=45)
    else:
        ax[1].remove()
    plt.tight_layout()
    plt.show()
else:
    print("No suitable numeric field selected for visualization.")

## 6. Conclusion

In this notebook, we've:
- Loaded and inspected metadata from the FAIR² Croissant-defined dataset via its schema URL.
- Enumerated all available record sets and fields with their `@id` values.
- Loaded tabular data from each record set, analyzed a sample numeric field, and grouped results by categorical fields where available.
- Visualized variable distributions and grouped summaries.

For further analysis, refer to the full schema and documentation of each record set using the record set and field `@id` as your primary lookup keys with `mlcroissant`.